# R3f retained-run read-only audit
Run All on CPU. Reads local Output or the pinned historical HF revision; no training, no remote writes, no Confirmation access. Missing historical candidate weights are reported, never reconstructed.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
SOURCE_SHA='9c86edc01bfcc154a43b447382f59136c53ed274'
ROOT=Path('/kaggle/working/r3f-readonly-code-'+SOURCE_SHA[:12])
OUTPUT=Path('/kaggle/working/r3f-output-'+SOURCE_SHA[:12])
ASSETS=Path('/kaggle/working/r3c-pilot-assets-d8a0fd0da03e')
def run(*args, **kwargs):
    print('[R3f]', *map(str,args), flush=True)
    if str(args[0])=='git':
        kwargs.setdefault('timeout',45)
    return subprocess.run(list(map(str,args)),check=True,**kwargs)
# Use installed PyTorch; only add the existing storage dependency if absent.
if importlib.util.find_spec('yaml') is None:
    run(sys.executable,'-m','pip','install','--quiet','pyyaml')
if importlib.util.find_spec('huggingface_hub') is None:
    run(sys.executable,'-m','pip','install','--quiet','huggingface_hub>=0.23,<2')
ROOT.mkdir(parents=True,exist_ok=True)
run('git','init','--quiet',ROOT)
remote=subprocess.run(['git','-C',str(ROOT),'remote','get-url','origin'],capture_output=True,text=True)
if remote.returncode:
    run('git','-C',ROOT,'remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
run('git','-C',ROOT,'config','remote.origin.promisor','true')
run('git','-C',ROOT,'config','remote.origin.partialclonefilter','blob:none')
run('git','-C',ROOT,'sparse-checkout','init','--no-cone')
run('git','-C',ROOT,'sparse-checkout','set','--no-cone',*['/scripts/run_r3f.py', '/pyproject.toml', '/src/', '/shared/python/', '/research/projects/cellular-language-models/engineering/kaggle-continuation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/', '/research/projects/cellular-language-models/experiments/006-path-first-clm/', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/', '/research/projects/continual-learning/context/04-continual-learning/scripts/research/', '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/'])
head=subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True)
if head.returncode or head.stdout.strip()!=SOURCE_SHA:
    run('git','-C',ROOT,'fetch','--depth=1','--filter=blob:none','origin',SOURCE_SHA)
    run('git','-C',ROOT,'checkout','--detach','FETCH_HEAD')
actual=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_SHA:
    raise RuntimeError('Source identity mismatch')
env=dict(os.environ,HF_HUB_DISABLE_PROGRESS_BARS='1',HF_HUB_ETAG_TIMEOUT='10',HF_HUB_DOWNLOAD_TIMEOUT='30',TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
# Bootstrap the existing runtime, then invoke only the read-only audit.
import runpy
import uuid
runpy.run_path(str(ROOT/'research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/run.py'), run_name='readonly_bootstrap')
from minicells_r3f.readonly import inspect, recover_trial
retained=Path('/kaggle/working/r3f-output-98cf8f6c214f')
if not (retained/'identity.json').is_file():
    print('Reading only two trial checkpoints from the pinned HF revision; no uploads.', flush=True)
    retained=recover_trial(Path('/kaggle/working')/('r3f-recovered-'+uuid.uuid4().hex))
result=inspect(retained, '/kaggle/working/r3f-readonly-audit')
for arm, evidence in result['arms'].items():
    print(arm, evidence['scan_status'])
    for proposal in evidence.get('verified_proposals', []):
        print({'identity': proposal.get('experiment_identity'), 'at_step': proposal['at_step'],
               'route_switch_margin': proposal.get('route_switch_margin'),
               'numeric_tolerance': proposal.get('numeric_tolerance'),
               'snapshot_fingerprint': proposal.get('snapshot_fingerprint')})
print('Save Version Output: /kaggle/working/r3f-readonly-audit. No training or optimizer updates.')
